# 03 - Stage B: GRPO (with DPO fallback) (skill 7)
Starts from the SFT adapter (merged into the base first so the KL anchor is the SFT policy).
Order: pilot (50 steps) -> read the printed completions -> full run -> selection -> full val ->
go/no-go. If GRPO is not working by the go/no-go deadline, run the DPO section; if that also does
not help, ship SFT and write it up honestly. Needs the SFT results from `02_sft` on the HF dataset repo.

In [ ]:
# 1. GPU + parameters
!nvidia-smi
import os, sys
REPO_URL = "https://github.com/YOUR_GITHUB_USER/tinyops"
HF_USER = "YOUR_HF_USERNAME"
DATA_REPO = f"{HF_USER}/tinyops-data"
SFT_REPO = f"{HF_USER}/tinyops-tiny-aya-sft"
FINAL_REPO = f"{HF_USER}/tinyops-tiny-aya"

In [ ]:
# 2. install, secrets, environment record
if not os.path.exists("tinyops"):
    !git clone {REPO_URL} tinyops
%cd tinyops
if sys.version_info >= (3, 12):
    PY = "python"
    !pip install -q -e ".[train]"
else:
    PY = "/content/venv/bin/python"
    !pip -q install uv
    !uv venv --python 3.12 /content/venv
    !uv pip install --python {PY} -q -e ".[train]"
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!{PY} -m tinyops.utils.envinfo
!{PY} -m tinyops.scenarios.build_dataset verify

In [ ]:
# 3. fetch the SFT adapter and the SFT/baseline results
from huggingface_hub import snapshot_download
snapshot_download(SFT_REPO, local_dir="runs/sft/final")
snapshot_download(DATA_REPO, repo_type="dataset", local_dir="hf_data",
                  allow_patterns=["baselines/metrics/*", "training_logs/sft/*"])
!mkdir -p results/metrics results/runs results/train_logs
!cp -n hf_data/baselines/metrics/* results/metrics/
!cp -n hf_data/training_logs/sft/sft_val.json results/metrics/ ; cp -n hf_data/training_logs/sft/sft_selection.json results/runs/
!{PY} -m tinyops.scenarios.build_dataset gold --split all --seed 1234

In [ ]:
# 4. PILOT: 50 steps. Every 25 steps the log prints 10 random completions with their reward breakdown.
!{PY} -m tinyops.train.grpo --config configs/grpo.yaml --set max_steps=50 --set output_dir=runs/grpo_pilot

**Read the printed completions before continuing.** Check: is the reward mean rising? Is KL
(`beta`, in `results/runs/grpo_train_run.json`) reasonable? Any reward hacking: instant submits,
evidence copied from the alert, refusing everything, runaway tool calls after `submit_diagnosis`?
Record what you saw in `docs/DECISIONS.md` (skill 7 exit gate).

In [ ]:
# 5. reward breakdown of the pilot at a glance
import json, pandas as pd
rows = [json.loads(l) for l in open("results/train_logs/grpo_rewards.jsonl")]
df = pd.DataFrame([{k: r[k] for k in ["total","diagnosis","format","schema","efficiency","grounding","safety"]} for r in rows])
df["batch"] = df.index // 8
df.groupby("batch").mean().round(3).tail(15)

In [ ]:
# 6. FULL RUN (resume-safe output_dir). Set max_steps to what the budget allows.
!{PY} -m tinyops.train.grpo --config configs/grpo.yaml --set max_steps=300

In [ ]:
# 7. selection rationale + curve, then the full val (180) with SFT+GRPO adapters
sel = json.load(open("results/runs/grpo_selection.json"))
print("keep SFT:", sel["keep_reference"], "| chosen:", sel["chosen"])
if sel["chosen"]:
    ckpt = sel["chosen"]["checkpoint"]
    !{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/tiny-aya-global --adapter runs/sft/final+{ckpt} \
        --split val --batch-size 8 --out results/episodes/grpo_val.jsonl
    !{PY} -m tinyops.train.stage_b go-no-go --stage-b results/metrics/grpo_val.json

## DPO fallback - only if go/no-go says Stage B is not working
Builds preference pairs from failed base/SFT rollouts on the **train** split (first divergence from
the gold episode), then trains with beta 0.1, lr 5e-6.

In [ ]:
RUN_DPO = False
if RUN_DPO:
    # failed rollouts on train scenarios: the base model and the SFT model
    !{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/tiny-aya-global --split train --limit 600 \
        --batch-size 8 --out results/episodes/base_train.jsonl --no-score
    !{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/tiny-aya-global --adapter runs/sft/final --split train \
        --limit 600 --batch-size 8 --out results/episodes/sft_train.jsonl --no-score
    !{PY} -m tinyops.train.dpo --config configs/dpo.yaml
    dsel = json.load(open("results/runs/dpo_selection.json"))
    if dsel["chosen"]:
        !{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/tiny-aya-global --adapter runs/sft/final+{dsel["chosen"]["checkpoint"]} \
            --split val --batch-size 8 --out results/episodes/dpo_val.jsonl

In [ ]:
# 8. choose the final model among SFT / SFT+GRPO / SFT+DPO by the plan's rule (val only)
cands = ["--candidate sft=results/metrics/sft_val.json"]
for name, f in [("sft+grpo", "grpo_val"), ("sft+dpo", "dpo_val")]:
    if os.path.exists(f"results/metrics/{f}.json"):
        cands.append(f"--candidate {name}=results/metrics/{f}.json")
!{PY} -m tinyops.train.stage_b final {" ".join(cands)}
print(open("results/runs/final_model_choice.json").read())

In [ ]:
# 9. push the final adapter, training logs and reward curves
from huggingface_hub import HfApi
choice = json.load(open("results/runs/final_model_choice.json"))["final"]
api = HfApi()
api.create_repo(FINAL_REPO, exist_ok=True)
src = {"sft+grpo": sel["chosen"]["checkpoint"] if sel["chosen"] else None}.get(choice) or "runs/sft/final"
api.upload_folder(folder_path=src, repo_id=FINAL_REPO, ignore_patterns=["optimizer.pt", "*.pth", "rng_state*"])
api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
for f in ["results/train_logs/grpo_rewards.jsonl", "results/train_logs/grpo_val_curve.csv", "results/runs/grpo_selection.json",
          "results/runs/grpo_train_run.json", "results/runs/stage_b_decision.json", "results/runs/final_model_choice.json",
          "runs/grpo/trajectories.jsonl", "runs/grpo/final/lexsi_provenance.json", "results/compute_log.csv"]:
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"training_logs/{os.path.basename(f)}", repo_id=DATA_REPO, repo_type="dataset")
print("final model:", choice, "->", FINAL_REPO, "(base: CohereLabs/tiny-aya-global; load with adapter 'sft_adapter+final_adapter' via HFPolicy)")